# Flip Patient OBJ Models

This notebook creates a flipped copy of all patient OBJ meshes.

In [3]:
from pathlib import Path

# Config
PROJECT_ROOT = Path.cwd().resolve()
INPUT_DIR = PROJECT_ROOT / 'data-processing' / 'cropped_data' / 'cropped'
OUTPUT_DIR = PROJECT_ROOT / 'data-processing' / 'cropped_flipped'
FLIP_AXIS = 'x'  # 'x', 'y', or 'z'
FLIP_NORMALS = True
REVERSE_FACE_WINDING = True  # keep outward-facing orientation after mirror
DRY_RUN = False

print('Input:', INPUT_DIR)
print('Output:', OUTPUT_DIR)
print('Flip axis:', FLIP_AXIS)
print('Flip normals:', FLIP_NORMALS, '\nReverse faces:', REVERSE_FACE_WINDING)


Input: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/notebooks/data-processing/cropped_data/cropped
Output: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/notebooks/data-processing/cropped_flipped
Flip axis: x
Flip normals: True 
Reverse faces: True


In [4]:
def _flip_xyz(x, y, z, axis: str):
    if axis == 'x':
        return -x, y, z
    if axis == 'y':
        return x, -y, z
    if axis == 'z':
        return x, y, -z
    raise ValueError('axis must be x, y, or z')

def _reverse_face_tokens(tokens):
    # tokens include leading 'f'
    if len(tokens) <= 2:
        return tokens
    return [tokens[0]] + list(reversed(tokens[1:]))

def flip_obj(in_path: Path, out_path: Path, axis: str, flip_normals: bool, reverse_faces: bool):
    out_lines = []
    with in_path.open('r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if line.startswith('v '):
                parts = line.strip().split()
                if len(parts) >= 4:
                    x, y, z = float(parts[1]), float(parts[2]), float(parts[3])
                    x, y, z = _flip_xyz(x, y, z, axis)
                    out_lines.append(f"v {x:.6f} {y:.6f} {z:.6f}\n")
                else:
                    out_lines.append(line)
            elif line.startswith('vn ') and flip_normals:
                parts = line.strip().split()
                if len(parts) >= 4:
                    x, y, z = float(parts[1]), float(parts[2]), float(parts[3])
                    x, y, z = _flip_xyz(x, y, z, axis)
                    out_lines.append(f"vn {x:.6f} {y:.6f} {z:.6f}\n")
                else:
                    out_lines.append(line)
            elif line.startswith('f ') and reverse_faces:
                tokens = line.strip().split()
                tokens = _reverse_face_tokens(tokens)
                out_lines.append(' '.join(tokens) + '\n')
            else:
                out_lines.append(line)

    if not DRY_RUN:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        out_path.write_text(''.join(out_lines), encoding='utf-8')

    return len(out_lines)


In [5]:
objs = sorted(INPUT_DIR.glob('*.obj'))
print('Found OBJ files:', len(objs))

if not objs:
    raise FileNotFoundError(f'No OBJ files found in {INPUT_DIR}')

written = 0
for p in objs:
    out_path = OUTPUT_DIR / p.name
    flip_obj(p, out_path, FLIP_AXIS, FLIP_NORMALS, REVERSE_FACE_WINDING)
    written += 1

print('Wrote flipped models:', written)


Found OBJ files: 0


FileNotFoundError: No OBJ files found in /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/notebooks/data-processing/cropped_data/cropped